> **Provenance.** This notebook is from Wen & Wang (2025), *Code for: Seismic wave propagation simulation with quantum computing*, Zenodo, [doi:10.5281/zenodo.14868887](https://doi.org/10.5281/zenodo.14868887), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Changes in this repository: outputs cleared, and this note plus the setup cell below added. The notebook uses the classical simulated-annealing sampler by default; the D-Wave QPU sampler is left commented out, as in the original.

In [ ]:
# Added for this repository: make the top-level `core` package importable
# when the notebook is run from the notebooks/ folder.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from core import stagger_FD_coeff as sfci
from core.utils import cholesky_solver
from neal import SimulatedAnnealingSampler
from dwave.system import DWaveSampler
from core.QUBO_solvers import IterSampleSolver
from collections import ChainMap
from matplotlib.ticker import MultipleLocator
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import time
from functools import wraps
pi = np.pi

In [ ]:
def call_back(x_mid, obj_mid):
    global it_num
    qa_res_dict[f"iter_{it_num}"] = x_mid
    it_num += 1
    obj_list.append(obj_mid)
    
def solve_coeff_QA():

    # Using the real Quantum Annealing Hardware Dwave System!
    # You should sign up a count and get your API.
    # sampler = DWaveSampler(
    #     token="YOUR API TOKEN",
    #     solver="Advantage_system6.4",
    #     region="na-west-1",
    # )  #

    # Using your computer to simulate the quantum annealing.
    sampler = SimulatedAnnealingSampler()  # for test

    qa_a0_dict = {}
    for M, freq_max in zip(M_set, freq_max_set):
        linear_sys = sfci.Stagger_FD_coeff_1D(v, h, tao, freq_max, L=20)
        sis = IterSampleSolver(
            sampler=sampler,
            x0=np.ones(M)*1e-2,  # initial vector of Delta a
            num_sample=1000,  # for real hardware device using 2000
            R=10,
            fixed_point=1,
            maxiter=3,  # max iteration in small iteration
            code_method="pn",
            resultMod="lowest_energy",
        )
        start = time.perf_counter()
        qa_a0_dict[f"QA M={M}"], qa_a0_dict[f"Iter M={M}"] = linear_sys.solve(
            M=M,
            coeff_vec=np.ones([M]).reshape(-1, 1),
            alpha=1.0,
            alpha_decay=0.1,
            beta=1.0,
            epsilon=1e-10,
            max_iter_num=max_iter_num,
            mode_of_construct="direct",
            solver=sis.solve,
            return_iter_num=True,
            call_back=call_back,
        )
        end = time.perf_counter()
        qa_time.append(end - start)
    return qa_a0_dict

In [ ]:
v = 4500
h = 10
tao = 0.001
freq_max_set = [190]
M_set = [10]
max_iter_num = 50
qa_time = []
qa_res_dict = {}
it_num = 1
obj_list = []
qa_a0_dict = solve_coeff_QA()
# result_dict = dict(ChainMap(qa_a0_dict, wang_a0_dict, tra_a0_dict))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.plot(obj_list[:-25], "o-", color="k")
ax.set_yscale("log")
ax.set_ylabel("Object Function", fontfamily="Arial", fontsize=12)
ax.set_xlabel("Iterations", fontfamily="Arial", fontsize=12)

ax.spines["left"].set_linewidth(1.5)
ax.spines["top"].set_linewidth(1.5)
ax.spines["right"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

ax.grid(visible=True, axis="both", which="major", linestyle="--", linewidth=0.5)

fig.savefig("Object_curve.png", dpi=600)
fig.savefig("Object_curve.pdf", dpi=600)

In [ ]:
def show_disp_curve(result_dict):
    fig, ax = plt.subplots(1, 1, figsize=(6, 5))

    cmap = cm.get_cmap('rainbow')  
    num_lines = len(result_dict)  
    colors = [cmap(i / num_lines) for i in range(num_lines)]
    for item, (label, coeff) in enumerate(result_dict.items()):
        linear_sys = sfci.Stagger_FD_coeff_1D(v, h, tao, L=20)
        sigma = linear_sys.cal_sigma(
            v * tao / h, np.linspace(pi / 1000, pi, 1000), coeff
        )

        ax.plot(
            np.linspace(pi / 1000, pi, 1000),
            sigma,
            label=f"{label}",
            color=colors[item]
        )

        ax.set_ylim([0.9, 1.05])
        ax.set_xlim([0, 3.142])
        ax.set_ylabel(
            "$\delta$",
            fontfamily="Arial",
            fontsize=20,
            math_fontfamily="stix",
        )
        ax.set_xlabel("$kh$", fontfamily="Arial", fontsize=20, math_fontfamily="stix")
        ax.set_xticks([0, 0.5, 1, 1.5, 2, 2.5, 3])
        ax.set_yticks([0.9, 0.95, 1.0, 1.05])

        ax.grid(visible=True, axis="both", which="major", linestyle="--", linewidth=0.5)

        ax.legend(
            title="",
            fontsize=12,
            loc="lower left",
            borderpad=0.5,
            borderaxespad=1.0,
            labelspacing=0.5,
            edgecolor="black",
            handlelength=2.0,
        )

        ax.spines["left"].set_linewidth(1.5)
        ax.spines["top"].set_linewidth(1.5)
        ax.spines["right"].set_linewidth(1.5)
        ax.spines["bottom"].set_linewidth(1.5)

    return fig, ax

In [ ]:
fig, ax = show_disp_curve({k:v for i, (k, v) in enumerate(qa_res_dict.items()) if i in [4, 5, 8, 9, 14, 19, 24]})
fig.savefig("disp_curve.png", dpi=600)
fig.savefig("disp_curve.pdf", dpi=600)